In [ ]:
!pip install pennylane

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.1/57.1 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 934.3/934.3 kB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 76.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 80.2 MB/s eta 0:00:00


In [ ]:
import pennylane as qml
import numpy as np

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
import sys

# Setting our constants
sys.path.append('..')

/usr/local/lib/python3.12/dist-packages/pennylane/__init__.py:209: RuntimeWarning: PennyLane is not yet compatible with JAX versions > 0.6.2. You have version 0.7.2 installed. Please downgrade JAX to 0.6.2 to avoid runtime errors using python -m pip install jax~=0.6.0 jaxlib~=0.6.0
  warnings.warn(


In [ ]:
!wget https://raw.githubusercontent.com/aifactory-team/AFCompetition/main/9245/train_X.npy
!wget https://raw.githubusercontent.com/aifactory-team/AFCompetition/main/9245/train_y.npy

--2025-12-15 04:36:19--  https://raw.githubusercontent.com/aifactory-team/AFCompetition/main/9245/train_X.npy
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 32896 (32K) [application/octet-stream]
Saving to: ‘train_X.npy’

train_X.npy         100%[===================>]  32.12K  --.-KB/s    in 0.01s   

2025-12-15 04:36:19 (3.02 MB/s) - ‘train_X.npy’ saved [32896/32896]

--2025-12-15 04:36:19--  https://raw.githubusercontent.com/aifactory-team/AFCompetition/main/9245/train_y.npy
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 256 [

In [ ]:
train_X = np.load("train_X.npy")
train_y = np.load("train_y.npy")

In [ ]:
import pennylane as qml
import torch
import torch.nn as nn
import torch.optim as optim
from pennylane import numpy as np
from torch.utils.data import TensorDataset, DataLoader

# ==========================================
# 1. Inference (Model Prediction) - 기존 유지
# ==========================================
def get_predictions(model, inputs):
    """Run inference on inputs using the trained model."""
    model.eval()
    with torch.no_grad():
        outputs = model(inputs)
        predicted_labels = torch.argmax(outputs, dim=1)
    return predicted_labels.cpu().numpy()

def data_to_tensor(X, y):
    tensor_X = torch.tensor(X, dtype=torch.complex64)
    tensor_y = torch.tensor(y, dtype=torch.long)
    return tensor_X, tensor_y

t_train_X, t_train_y = data_to_tensor(train_X, train_y)
train_dataset = TensorDataset(t_train_X, t_train_y)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

In [ ]:
import pennylane as qml
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import json

# ==========================================
# Configuration
# ==========================================
N_QUBITS = 8
N_LAYERS = 5  # Adjustable
BATCH_SIZE = 4
EPOCHS = 200
LEARNING_RATE = 0.05
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ==========================================
# Data Loading
# ==========================================
def load_data():
    """Load training data from numpy files"""
    train_X = np.load("train_X.npy")
    train_y = np.load("train_y.npy")

    tensor_X = torch.tensor(train_X, dtype=torch.complex64)
    tensor_y = torch.tensor(train_y, dtype=torch.long)

    return tensor_X, tensor_y

# ==========================================
# Quantum Circuit Definition
# ==========================================
dev = qml.device("default.qubit", wires=N_QUBITS)

def quantum_layer(params, layer_idx, n_qubits):
    """
    Single layer of quantum circuit with rotation + entanglement

    Args:
        params: Shape (n_layers, 3, n_qubits)
        layer_idx: Current layer index
        n_qubits: Number of qubits
    """
    # Rotation gates for each qubit
    for i in range(n_qubits):
        qml.Rot(
            params[layer_idx, 0, i],
            params[layer_idx, 1, i],
            params[layer_idx, 2, i],
            wires=i
        )

    # Entanglement (CNOT ladder)
    for i in range(n_qubits - 1):
        qml.CNOT(wires=[i, i + 1])

def quantum_classifier(params, n_layers, n_qubits):
    """
    Complete quantum classifier circuit

    Args:
        params: Trainable parameters, shape (n_layers, 3, n_qubits)
        n_layers: Number of layers
        n_qubits: Number of qubits
    """
    params = params.reshape(n_layers, 3, n_qubits)

    # Apply all layers except the last one (with entanglement)
    for layer in range(n_layers - 1):
        quantum_layer(params, layer, n_qubits)

    # Final layer (only rotations, no entanglement)
    for i in range(n_qubits):
        qml.Rot(
            params[-1, 0, i],
            params[-1, 1, i],
            params[-1, 2, i],
            wires=i
        )

@qml.qnode(dev, interface='torch')
def quantum_circuit(state, params, n_layers, n_qubits, measure_wires):
    """
    Full quantum circuit with state preparation and measurement

    Args:
        state: Input quantum state
        params: Circuit parameters
        n_layers: Number of layers
        n_qubits: Number of qubits
        measure_wires: Wires to measure
    """
    # State preparation
    qml.StatePrep(state, wires=range(n_qubits))

    # Quantum classifier
    quantum_classifier(params, n_layers, n_qubits)

    # Measurement
    return qml.probs(wires=measure_wires)

# ==========================================
# PyTorch Model Wrapper
# ==========================================
class QuantumNeuralNetwork(nn.Module):
    def __init__(self, n_qubits, n_layers, measure_wires=[6, 7]):
        super().__init__()
        self.n_qubits = n_qubits
        self.n_layers = n_layers
        self.measure_wires = measure_wires
        self.total_params = n_layers * 3 * n_qubits

        # Initialize parameters with small random values
        torch.manual_seed(42)
        self.params = nn.Parameter(
            torch.randn(self.total_params) * 0.01
        )

    def forward(self, x):
        return quantum_circuit(x, self.params, self.n_layers,
                              self.n_qubits, self.measure_wires)

# ==========================================
# Loss Function
# ==========================================
def quantum_cross_entropy_loss(probs, labels):
    """
    Cross-entropy loss for quantum classifier

    Args:
        probs: Predicted probabilities (batch_size, n_classes)
        labels: True labels (batch_size,)
    """
    # One-hot encode labels
    one_hot = torch.nn.functional.one_hot(labels, num_classes=probs.shape[1])

    # Normalize probabilities
    probs = probs / torch.sum(probs, dim=1, keepdim=True)

    # Add small epsilon to avoid log(0)
    eps = 1e-10
    probs = torch.clamp(probs, min=eps, max=1.0)

    # Calculate cross-entropy loss
    loss = -torch.sum(one_hot * torch.log(probs), dim=1)

    return torch.mean(loss)

# ==========================================
# Training Function
# ==========================================
def train_model(model, train_loader, epochs, lr, device):
    """Train the quantum neural network"""
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    loss_history = []
    acc_history = []

    print(f"Training QNN with {model.total_params} parameters...")
    print(f"Using device: {device}")

    for epoch in range(epochs):
        total_loss = 0
        correct = 0

        for batch_X, batch_y in train_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            # Forward pass
            optimizer.zero_grad()
            predictions = model(batch_X)

            # Calculate loss
            loss = quantum_cross_entropy_loss(predictions, batch_y)

            # Backward pass
            loss.backward()
            optimizer.step()

            # Track metrics
            total_loss += loss.item()
            predicted_classes = torch.argmax(predictions, dim=1)
            correct += (predicted_classes == batch_y).sum().item()

        # Calculate average metrics
        avg_loss = total_loss / len(train_loader)
        avg_acc = correct / len(train_loader.dataset)

        loss_history.append(avg_loss)
        acc_history.append(avg_acc)

        # Print progress
        if (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch+1:3d} | Loss: {avg_loss:.4f} | Acc: {avg_acc:.4f}")

    return loss_history, acc_history

# ==========================================
# QASM Export Function
# ==========================================
def export_to_qasm(model, filename="submission.json"):
    """
    Export trained model to OpenQASM format

    Args:
        model: Trained quantum model
        filename: Output JSON filename
    """
    # Extract parameters
    params = model.params.detach().cpu().numpy()

    # Define circuit for QASM conversion (no StatePrep or measurement)
    @qml.qnode(dev, interface='torch')
    def circuit_for_qasm(params):
        quantum_classifier(params, model.n_layers, model.n_qubits)

    # Generate QASM
    qasm_data = qml.to_openqasm(circuit_for_qasm, measure_all=False)(params)

    # Save to JSON
    output_data = {
        "qasm": qasm_data,
        "measurements": model.measure_wires
    }

    with open(filename, "w") as f:
        json.dump(output_data, f, indent=2)

    print(f"\n✅ QASM exported to '{filename}'")
    print(f"   - Measurement qubits: {model.measure_wires}")
    print(f"   - QASM length: {len(qasm_data)} characters")

# ==========================================
# Evaluation Function
# ==========================================
def evaluate_model(model, data_loader, device):
    """Evaluate model accuracy"""
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_X, batch_y in data_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            predictions = model(batch_X)
            predicted_classes = torch.argmax(predictions, dim=1)

            correct += (predicted_classes == batch_y).sum().item()
            total += batch_y.size(0)

    accuracy = correct / total
    return accuracy

# ==========================================
# Main Execution
# ==========================================
def main():
    """Main training pipeline"""
    print("="*60)
    print("Quantum Circuit Classifier - Optimized Version")
    print("="*60)

    # Load data
    print("\n📊 Loading data...")
    train_X, train_y = load_data()
    train_dataset = TensorDataset(train_X, train_y)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

    print(f"   - Training samples: {len(train_dataset)}")
    print(f"   - Batch size: {BATCH_SIZE}")
    print(f"   - Input shape: {train_X[0].shape}")

    # Initialize model
    print(f"\n🔧 Initializing model...")
    model = QuantumNeuralNetwork(
        n_qubits=N_QUBITS,
        n_layers=N_LAYERS,
        measure_wires=[6, 7]
    )
    print(f"   - Qubits: {N_QUBITS}")
    print(f"   - Layers: {N_LAYERS}")
    print(f"   - Total parameters: {model.total_params}")

    # Train model
    print(f"\n🚀 Starting training...")
    loss_history, acc_history = train_model(
        model, train_loader, EPOCHS, LEARNING_RATE, DEVICE
    )

    # Final evaluation
    print(f"\n📈 Final Results:")
    final_acc = evaluate_model(model, train_loader, DEVICE)
    print(f"   - Training Accuracy: {final_acc:.4f}")
    print(f"   - Final Loss: {loss_history[-1]:.4f}")

    # Export to QASM
    export_to_qasm(model, "optimized_submission.json")

    print("\n✅ Training complete!")
    print("="*60)

    return model, loss_history, acc_history

if __name__ == "__main__":
    model, loss_history, acc_history = main()

In [ ]:
from google.colab import files
files.download('baseline.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>